In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import json

results = []
for i in range(60): # number of simulations
    !./week1
    with open("data.json") as jsondata:
        params = json.load(jsondata); seed = params["seed"]
        T = params["T"]; C = params["C"]; P = params["P"]

    VTXdf = pd.read_csv("vtx.csv")
    dumbdf = pd.read_csv("dumb.csv")
    pricedf = pd.read_csv("price.csv")
    demanddf = pd.read_csv("demand.csv")
    dp_rev = VTXdf[(VTXdf['t'] == 1) & (VTXdf['x'] == C-1)]['VTX'].values[0]

    row = dict(zip(dumbdf['p'], dumbdf['rev']))
    row['dp_rev'] = dp_rev
    row['best_diff'] = dp_rev - dumbdf['rev'].max()
    row['seed'] = seed
    results.append(row)

output = pd.DataFrame(results)
output = output.sort_values('best_diff', ascending=False)
output.to_excel("output.xlsx", index=False)



In [ ]:
def makePlots():
    fig, (f1, f2, f3) = plt.subplots(1, 3, figsize=(16, 5))
    fig.suptitle(f"seed={seed} | dp={dp_rev:.2f} | C = {C}")
    for t in range(1, T, C):
        f1.plot(VTXdf[VTXdf['t'] == t]['x'], VTXdf[VTXdf['t'] == t]['VTX'], label=f't={t}')
        f2.plot(pricedf[pricedf['t'] == t]['x'], pricedf[pricedf['t'] == t]['opt'], label=f't={t}')
    f3.plot(demanddf['t'], demanddf['alpha'], label='alpha')
    f3.plot(demanddf['t'], demanddf['gamma'], label='gamma')
        
    f1.set_xlabel('x'); f2.set_xlabel('x');  f3.set_xlabel('t')
    f1.set_ylabel('VTX'); f2.set_ylabel('Optimal Price')
    f1.legend(); f2.legend(); f3.legend()

In [52]:
def makeExcel():
    pricedf.pivot(index='x', columns='t', values='opt').to_excel(f"PRICEsim{i}.xlsx", index=False)
    dumbdf.pivot(index='price', columns='t', values='revenue').to_excel(f"dumbsim{i}.xlsx", index=False)
    VTXdf.pivot(index='x', columns='t', values='VTX').to_excel(f"VTXsim{i}.xlsx", index=False)